In [1]:
import requests
import json
import pandas as pd
import os
from tqdm import tqdm

In [2]:
class whoAmI:
    def __init__(self):
        self.getDim='https://ghoapi.azureedge.net/api/Dimension'
        self.getDimValues='https://ghoapi.azureedge.net/api/DIMENSION/COUNTRY/DimensionValues'
        self.getIndValues='https://ghoapi.azureedge.net/api/Indicator'
        self.getIndFilterVal='https://ghoapi.azureedge.net/api/Indicator?$filter=contains(IndicatorName,'
        self.getIndFilterSpecVal='https://ghoapi.azureedge.net/api/Indicator?$filter=IndicatorName eq '
        self.getIndData='https://ghoapi.azureedge.net/api/'
urls = whoAmI()

In [5]:
DATASET_SAVED = "../../data/raw_official_v2"
URL_INDICATORS = "../../data/urls"

In [10]:
# Khởi tạo thư mục nếu chưa tồn tại
os.makedirs(DATASET_SAVED, exist_ok=True)
os.makedirs(URL_INDICATORS, exist_ok=True)

In [20]:
# Lấy các bộ dữ liệu đã có
existed = set()
for file in os.listdir(DATASET_SAVED):
    indicator = file.split('.')[0]
    existed.add(indicator)

In [21]:
dataset_indicators = set()
for file in os.listdir(URL_INDICATORS):
    with open(URL_INDICATORS + '/' + file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line in existed:
                continue
            dataset_indicators.add(line)
print("Các indicators:", dataset_indicators)

Các indicators: {'WHS6_15'}


In [7]:
def request_data(indicator):
    resp = requests.get(urls.getIndData + '/' + indicator)
    data = json.loads(resp.content)["value"]
    fields = ['ParentLocationCode', 'SpatialDim', 'Value', 'NumericValue', 'TimeDimensionBegin', 'TimeDimensionEnd', 'TimeDimensionValue', 'TimeDimType', 'TimeDim', 'IndicatorCode', 'Date']

    records = []
    for row in data:
        _tempo = dict()
        for field in fields:
            _tempo[field] = row[field]
        records.append(_tempo)
    
    with open(DATASET_SAVED + '/' + indicator + '.json', "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [8]:
# Thử chương trình lấy một bộ dữ liệu
# request_data("SA_0000001403")

In [14]:
# Hàm lấy hết dữ liệu
def retrieve_all():
    for indicator in tqdm(dataset_indicators):
        try:
            request_data(indicator)
        except Exception as e:
            print("Error:", e)

In [ ]:
#retrieve_all()

100%|██████████| 1/1 [00:01<00:00,  1.97s/it]

Error: Expecting value: line 1 column 1 (char 0)


 18%|█▊        | 102/553 [03:45<28:57,  3.85s/it]

Error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


 19%|█▊        | 103/553 [03:55<42:38,  5.69s/it]

Error: HTTPSConnectionPool(host='ghoapi.azureedge.net', port=443): Max retries exceeded with url: /api//E12_brand_sharing (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1007)')))


 19%|█▉        | 105/553 [04:05<40:21,  5.41s/it]

Error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


100%|██████████| 553/553 [30:26<00:00,  3.30s/it]  


In [19]:
retrieve_all()

 75%|███████▌  | 3/4 [00:17<00:04,  4.22s/it]

Error: Expecting value: line 1 column 1 (char 0)


100%|██████████| 4/4 [00:18<00:00,  4.59s/it]


In [2]:
# Biến chỉ định lưu trữ
CONCAT_GROUP = "../../data/raw_concat"
os.makedirs(CONCAT_GROUP, exist_ok=True)

In [6]:
# Tiến hành tổ chức và nhóm lại dữ liệu vào file gốc theo các nhãn
def prepare_data_through_group(group):
    # Group là nhãn dữ liệu, tên thư mục
    target = group.split('.')[0]
    objectives = set()

    with open(URL_INDICATORS + '/' + group, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            objectives.add(line)
    
    array_data = []
    for file in tqdm(os.listdir(DATASET_SAVED)):
        _label = file.split('.')[0]
        if _label in objectives:
            # Đúng nhãn dữ liệu, tiến hành load ra và ghép lại thành một file hoàn chỉnh
            with open(DATASET_SAVED + '/' + file, "rb") as f:
                data = json.load(f)
                
                # Tiến hành nối dữ liệu vào file
            array_data = array_data + data
                
    concated_result = CONCAT_GROUP + '/' + target + '.json'
    
    print("Tổng số điểm dữ liệu là:", len(array_data))
    with open(concated_result, "w", encoding="utf-8") as f:
        json.dump(array_data, f, ensure_ascii=False, indent=2)
    print("Hoàn thành xử lý:", target)

In [13]:
# Chạy xử lí chỉ số BMI
prepare_data_through_group("BMI.txt")

100%|██████████| 552/552 [00:01<00:00, 543.16it/s]


Tổng số điểm dữ liệu là: 453077
Hoàn thành xử lý: BMI


In [24]:
# CHạy xử lí chỉ số Diabetes
prepare_data_through_group("diabetes.txt")

100%|██████████| 552/552 [00:02<00:00, 213.70it/s]


Tổng số điểm dữ liệu là: 211564
Hoàn thành xử lý: diabetes


In [15]:
# Chạy xử lí tiêu thụ cồn
prepare_data_through_group("alcohol_consumption.txt")

100%|██████████| 552/552 [00:04<00:00, 124.64it/s] 


Tổng số điểm dữ liệu là: 301920
Hoàn thành xử lý: alcohol_consumption


In [7]:
# Chạy xử lí air_pollution
prepare_data_through_group("air_pollution.txt")

  0%|          | 0/552 [00:00<?, ?it/s]

100%|██████████| 552/552 [00:03<00:00, 156.39it/s]


Tổng số điểm dữ liệu là: 491076
Hoàn thành xử lý: air_pollution


In [17]:
# Chạy xử lí cholesterol
prepare_data_through_group("cholesterol.txt")

100%|██████████| 552/552 [00:00<00:00, 1286.68it/s]


Tổng số điểm dữ liệu là: 141804
Hoàn thành xử lý: cholesterol


In [18]:
# Chạy xử lí Glucose
prepare_data_through_group("glucose.txt")

100%|██████████| 552/552 [00:00<00:00, 10795.14it/s]

Tổng số điểm dữ liệu là: 21630


Hoàn thành xử lý: glucose


In [25]:
# Chạy xử lí hoạt động thể chất
prepare_data_through_group("physical_activities.txt")

100%|██████████| 552/552 [00:00<00:00, 1623.85it/s]


Tổng số điểm dữ liệu là: 35523
Hoàn thành xử lý: physical_activities


In [20]:
# Chạy xử lí tiếp cận cơ sở y tế
prepare_data_through_group("infrastructure.txt")

100%|██████████| 552/552 [00:00<00:00, 7696.94it/s]

Tổng số điểm dữ liệu là: 8989


Hoàn thành xử lý: infrastructure


In [21]:
# Chạy xử lí thuốc lá
prepare_data_through_group("tobacco.txt")

100%|██████████| 552/552 [00:09<00:00, 56.36it/s] 


Tổng số điểm dữ liệu là: 704483
Hoàn thành xử lý: tobacco


In [22]:
# Chạy xử lí bệnh tim mạch
prepare_data_through_group("cardiovascular_diseases.txt")

100%|██████████| 552/552 [00:00<00:00, 737.96it/s]


Tổng số điểm dữ liệu là: 236876
Hoàn thành xử lý: cardiovascular_diseases
